# In this notebook we try to diagonalize the fit some numerical data on which noise was added

The ATS Hamiltonian reads $$H = \omega_a a^\dagger a + \omega_b b^\dagger b - 2E_J\cos(\phi_\Sigma)\cos(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta) + 2\Delta E_J\sin(\phi_\Sigma)\sin(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta)$$

We can set $\phi_\Sigma = 0$ so that $$H = \omega_a a^\dagger a + \omega_b b^\dagger b - 2E_J\cos(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta),$$
we retrieve the Fluxonium Hamiltonian.

In [ ]:
import dynamiqs as dq
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import Plot_setting
from Plot_setting import *
# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 3.6 * 2 * np.pi
w_b = 6.6 * 2 * np.pi

phi_a = 0.11
phi_b = 0.2

E_J = 26 * 2 * np.pi 

# drive/pump base params
epsilon_0_base = 0.08
g2 = sp.special.jv(1, epsilon_0_base) * E_J * phi_a**2 * phi_b

In [ ]:
n_a = 20
n_b = 10
a = dq.destroy(n_a)
b = dq.destroy(n_b)
varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()))

H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b))

In [ ]:
phi_delta_list = np.linspace(0, 2*np.pi, 21)
transitions_list = []

for phi_delta in phi_delta_list:
    phi_op = varphi_a_op_tot + varphi_b_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op)
    H = H.to_numpy()

    evals, evecs = np.linalg.eigh(H)
    evals = np.real(evals)
    evals = np.sort(evals)

    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:5])

transitions_list_bare = np.array(transitions_list)
transitions_list_bare+=np.random.randn(transitions_list_bare.shape[0],transitions_list_bare.shape[1])

In [ ]:
for i in range(1, transitions_list_bare.shape[1]):
    plt.scatter(phi_delta_list, transitions_list_bare[:, i] / (2*np.pi))

plt.xlabel(r'$\phi_\Delta$')
plt.ylabel('Transition frequency')
plt.show()

In [ ]:
def compute_transitions(params, phi_delta_list, n_levels=5):
    phi_a, phi_b = params
    varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b))
    varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()))

    transitions_list = []

    for phi_delta in phi_delta_list:
        phi_op = (
            varphi_a_op_tot
            + varphi_b_op_tot
            + phi_delta * dq.eye_like(varphi_a_op_tot)
        )

        H = H_base - 2 * E_J * dq.cosm(phi_op)
        H = H.to_numpy()

        evals, _ = np.linalg.eigh(H)
        evals = np.sort(np.real(evals))

        transitions = evals - evals[0]
        transitions_list.append(transitions[:n_levels])

    return np.array(transitions_list)
def residuals(params, phi_delta_list, data):
    model = compute_transitions(params, phi_delta_list, n_levels=data.shape[1])
    return (model - data).ravel()

params0 = [phi_a-0.05, phi_b+0.1]   # initial guess
result = least_squares(residuals, params0, args=(phi_delta_list, transitions_list_bare))
params_fit = result.x

print("Fitted parameters:", params_fit)
print("Cost:", result.cost)

In [ ]:
phi_delta_list_fit = np.linspace(0, 2*np.pi, 101)
transitions_fit = compute_transitions(params_fit, phi_delta_list, n_levels=transitions_list_bare.shape[1])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharex=True)
ax1, ax2 = axes

for i in range(1,transitions_list_bare.shape[1]):
    ax1.scatter(phi_delta_list, transitions_list_bare[:, i]/(2*np.pi), alpha=0.6,
                label=f"fit: "+ r"$\varphi_a=$"+str(round(phi_a,3)) + ", " + r"$\varphi_b=$"+str(round(phi_b,3)) if i == 1 else None)
    ax1.plot(phi_delta_list, transitions_fit[:, i]/(2*np.pi), '-', lw=2,
            label=f"fit: "+ r"$\varphi_a=$"+str(round(params_fit[0],3)) + ", " + r"$\varphi_b=$"+str(round(params_fit[1],3)) if i == 1 else None)

    ax2.plot(phi_delta_list, (transitions_list_bare[:, i] - transitions_fit[:, i])/(2*np.pi), 'o-', alpha=0.8)

ax1.set_ylabel("Transition frequency")
ax2.set_xlabel(r'$\phi_\Delta$')
ax2.set_ylabel("Residuals")

ax1.legend()
plt.tight_layout()
plt.show()

# We now add a third mode in the ATS 
This mode will have high frequency $\omega_c/2\pi=15$ GHz, and a large $\varphi_{zpf}$, let's take $\varphi_{c} = 10\varphi_a$

$$H = \omega_a a^\dagger a + \omega_b b^\dagger b + \omega_c c^\dagger c - 2E_J\cos\bigl[\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\varphi_c (c+c^\dagger)+\phi_\Delta\bigr],$$

In [ ]:
w_c = 15 * 2 * np.pi
phi_c = phi_b*2

n_a = 20
n_b = 6
n_c = 5
a = dq.destroy(n_a)
b = dq.destroy(n_b)
c = dq.destroy(n_c)
varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b), dq.eye(n_c))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()), dq.eye(n_c))
varphi_c_op_tot = dq.tensor(dq.eye(n_a), dq.eye(n_b), phi_c*(c + c.dag()))

H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), dq.eye(n_b),  w_c * dq.number(n_c))

In [ ]:
phi_delta_list = np.linspace(np.pi-.5, np.pi+.5, 21)
transitions_list = []

for phi_delta in phi_delta_list:
    phi_op = varphi_a_op_tot + varphi_b_op_tot + varphi_c_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op)
    H = H.to_numpy()

    evals, evecs = sp.sparse.linalg.eigsh(H, k=5, which='SM', tol = 0, maxiter=1e6)
    evals = np.real(evals)
    evals = np.sort(evals)
    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:10])

transitions_list_three_mode = np.array(transitions_list)
transitions_list_three_mode+=np.random.randn(transitions_list_three_mode.shape[0],transitions_list_three_mode.shape[1])/1

In [ ]:
for i in range(1, transitions_list_three_mode.shape[1]):
    plt.scatter(phi_delta_list, transitions_list_three_mode[:, i] / (2*np.pi))

plt.xlabel(r'$\phi_\Delta$')
plt.ylabel('Transition frequency')
plt.show()

In [ ]:
def compute_transitions_three_modes(params, phi_delta_list, n_levels=5):
    phi_a, phi_b, phi_c = params
    varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b), dq.eye(n_c))
    varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()), dq.eye(n_c))
    varphi_c_op_tot = dq.tensor(dq.eye(n_a), dq.eye(n_b), phi_c*(c + c.dag()))

    transitions_list = []

    for phi_delta in phi_delta_list:
        phi_op = varphi_a_op_tot + varphi_b_op_tot + varphi_c_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
        H = H_base - 2 * E_J * dq.cosm(phi_op)
        H = H.to_numpy()

        evals, _ = np.linalg.eigh(H)
        evals = np.sort(np.real(evals))

        transitions = evals - evals[0]
        transitions_list.append(transitions[:n_levels])

    return np.array(transitions_list)
def residuals(params, phi_delta_list, data):
    model = compute_transitions_three_modes(params, phi_delta_list, n_levels=data.shape[1])
    return (model - data).ravel()

params0 = [phi_a-0.05, phi_b+0.1, phi_c+0.21]   # initial guess

lower_bounds = [0.0, 0.0, 0.0]
upper_bounds = [1, 1, 1]

result = least_squares(
    residuals,
    x0=params0,
    bounds=(lower_bounds, upper_bounds),
    args=(phi_delta_list, transitions_list_three_mode),
)
params_fit_three_modes = result.x

print("Fitted parameters:", params_fit_three_modes)
print("Cost:", result.cost)

In [ ]:
phi_delta_list_fit = np.linspace(0, 2*np.pi, 101)
transitions_fit_three_modes = compute_transitions_three_modes(params_fit_three_modes, phi_delta_list, n_levels=transitions_list_three_mode.shape[1])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharex=True)
ax1, ax2 = axes

for i in range(1,transitions_list_bare.shape[1]):
    ax1.scatter(phi_delta_list, transitions_list_three_mode[:, i]/(2*np.pi), alpha=0.6,
                label=f"fit: "+ r"$\varphi_a=$"+str(round(phi_a,3)) + ", " + r"$\varphi_b=$"+str(round(phi_b,3)) + ", " + r"$\varphi_c=$"+str(round(phi_c,3)) if i == 1 else None)
    ax1.plot(phi_delta_list, transitions_fit_three_modes[:, i]/(2*np.pi), '-', lw=2,
            label=f"fit: "+ r"$\varphi_a=$"+str(round(params_fit_three_modes[0],3)) + ", " + r"$\varphi_b=$"+str(round(params_fit_three_modes[1],3)) + ", " + r"$\varphi_c=$"+str(round(params_fit_three_modes[2],3)) if i == 1 else None)

    ax2.plot(phi_delta_list, (transitions_list_three_mode[:, i] - transitions_fit_three_modes[:, i])/(2*np.pi), 'o-', alpha=0.8)

ax1.set_ylabel("Transition frequency")
ax2.set_xlabel(r'$\phi_\Delta$')
ax2.set_ylabel("Residuals")

ax1.legend()
plt.tight_layout()
plt.show()